# Exercise 2 - Inverse Perturbation (Two States)

Objective: predict perturbation from two consecutive states with
`dataset_mode="inverse_perturbation_two_states"`.

This mode uses `[X_t, X_{t+1}] -> P_t`, so the MLP shape is
`(2 * expression_dim, hidden_d, perturbation_dim)` (here `(100, hidden_d, 8)`).


In [1]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path.cwd() / "src"))

from perturbation_pipeline import (
    evaluate_predictions_by_timepoint,
    load_experiment_data,
    merge_train_val_splits,
    split_dataset,
    test,
    train_with_model_selection,
)

In [2]:
DATA_DIR = Path("simulated_data_for_interview_exercise")
adata = load_experiment_data(DATA_DIR)

print("adata shape:", adata.shape)
print("expression dim:", adata.n_vars)
print("perturbation dim:", adata.obsm["perturbation"].shape[1])
print("timepoints:", sorted(adata.obs["round"].unique().tolist()))

adata shape: (48070, 50)
expression dim: 50
perturbation dim: 8
timepoints: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [3]:
splits = split_dataset(
    adata=adata,
    test_timepoints=(9, 10),
    val_fraction=0.2,
    random_state=42,
    dataset_mode="inverse_perturbation_two_states", # much better than doing a-b even given the same number of parameters.. 
)

for split_name in ["train", "val", "test"]:
    x_split, y_split = splits[split_name]
    print(split_name, "X:", x_split.shape, "y:", y_split.shape)

input_dim = splits["train"][0].shape[1]
output_dim = splits["train"][1].shape[1]

train X: (26920, 100) y: (26920, 8)
val X: (6729, 100) y: (6729, 8)
test X: (9614, 100) y: (9614, 8)


In [4]:
# Keep this example simple: choose among a small set of models
train_pool = merge_train_val_splits(splits)

selected = train_with_model_selection(
    train_data=train_pool,
    candidate_models=("mlp", "linear_regression"),
    n_splits=5,
    random_state=42,
    mlp_hidden_dim=[32, 32],
    mlp_max_iter=300,
)

print("Best model:", selected["best_model_name"])

summary_rows = []
for model_name, result in selected["cv_results"].items():
    row = {"model": model_name}
    row.update({f"mean_{k}": v for k, v in result["mean_metrics"].items()})
    row.update({f"std_{k}": v for k, v in result["std_metrics"].items()})
    summary_rows.append(row)

pd.DataFrame(summary_rows).sort_values("mean_rmse").reset_index(drop=True)

Best model: mlp


,model,mean_mse,mean_rmse,mean_mae,mean_r2,std_mse,std_rmse,std_mae,std_r2
0,mlp,0.131703,0.362904,0.257075,0.683235,0.001426,0.001962,0.001509,0.003241
1,linear_regression,0.183464,0.428324,0.338270,0.558696,0.001389,0.001622,0.000990,0.003461


In [5]:
test_output = test(selected["model"], splits["test"])
print("Test metrics:", test_output["metrics"])

per_tp_metrics = evaluate_predictions_by_timepoint(
    prediction=test_output["prediction"],
    ground_truth=splits["test"][1],
    timepoints=splits["timepoints"]["test"],
)
per_tp_metrics

Test metrics: {'mse': 0.21909424662590027, 'rmse': 0.46807504379735976, 'mae': 0.3509249687194824, 'r2': 0.47842466831207275}


{'overall': {'mse': 0.21909424662590027,
  'rmse': 0.46807504379735976,
  'mae': 0.3509249687194824,
  'r2': 0.47842466831207275},
 'timepoint_9': {'mse': 0.21619293093681335,
  'rmse': 0.464965515857696,
  'mae': 0.3477497398853302,
  'r2': 0.4824966788291931},
 'timepoint_10': {'mse': 0.22199584543704987,
  'rmse': 0.47116435077056695,
  'mae': 0.3541008234024048,
  'r2': 0.4736221432685852}}